# Evaluation Results of RAG

First, import all required libraries.

In [1]:
from ResearchRAG.ingestion.loaders import parse_pdf_folder
from ResearchRAG.ingestion.chunking import split_text
from ResearchRAG.embedding.embeddings import build_embedding_model
from ResearchRAG.embedding.vectorstore import build_database, save_faiss_index, load_faiss_index
from ResearchRAG.retrieval.retriever import build_similarity_retriever
from ResearchRAG.evaluation.evaluation import load_eval_data, evaluate_retrieval, summarize_retrieval_results, save_evaluation_results
from ResearchRAG.retrieval.reranking import build_rerank_retriever
from ResearchRAG.config import RERANK_BASE_K, RAW_PDF_DIR
from dotenv import load_dotenv

load_dotenv()


C:\Users\pstfc\PycharmProjects\GenAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

Load embedding model:

In [2]:
embedding_model = build_embedding_model("miniLM")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7348.29it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 1.1.A Build Database

In [4]:
documents = parse_pdf_folder(RAW_PDF_DIR)
chunked_documents = split_text(documents)
vectorstore = build_database(chunked_documents, embedding_model)
save_faiss_index(vectorstore, "faiss_miniLM")

### 1.1.B Load Database

In [3]:
vectorstore = load_faiss_index("faiss_miniLM", embedding_model)

## 1. Retriever Only
First, let's evaluate the RAG with retriever only.

### 1.2. Build Retriever

In [4]:
base_retriever = build_similarity_retriever(vectorstore, k=RERANK_BASE_K)

In [5]:
eval_data = load_eval_data("Evaluation_QA_2.json")

In [6]:
base_results = evaluate_retrieval(eval_data, base_retriever)
base_summary = summarize_retrieval_results(base_results)

print(base_summary)

save_evaluation_results(
    base_results,
    base_summary,
    "baseline_retrieval_results.json"
)

{'num_questions': 18, 'hit_rate': 0.4444444444444444, 'average_recall': 0.38333333333333336}


## 2. With Reranker

Now, let's add reranker to the pipeline and test the results.

In [5]:
rerank_retriever = build_rerank_retriever(base_retriever)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7441.11it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
rerank_results = evaluate_retrieval(eval_data, rerank_retriever)
rerank_summary = summarize_retrieval_results(rerank_results)

print("Rerank summary:", rerank_summary)

save_evaluation_results(
    rerank_results,
    rerank_summary,
    "rerank_retrieval_results.json"
)

## Evaluate using RAGAS

In [6]:
from src.ResearchRAG.evaluation.evaluation import run_experiment, build_ragas_dataset
from src.ResearchRAG.generation.llms import build_llm
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

C:\Users\pstfc\AppData\Local\Temp\ipykernel_25200\1168822993.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\pstfc\AppData\Local\Temp\ipykernel_25200\1168822993.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
C:\Users\pstfc\AppData\Local\Temp\ipykernel_25200\1168822993.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from

In [7]:
dataset = build_ragas_dataset(eval_data)

In [8]:
rag_llm = build_llm("mistral")
evaluator_llm = build_llm("llama3")

In [9]:
results = await run_experiment.arun(
    dataset,
    retriever=base_retriever,
    rag_llm=rag_llm,
    evaluator_llm=evaluator_llm,
    experiment_name = "exp1"
)
results.to_pandas()

Running experiment: 100%|██████████| 18/18 [01:54<00:00,  6.39s/it]

""
